# Email Finder — YouTube list (LITE, HTTP-only, no Playwright)

Built for one goal: **find an email by discovering the channel's website and
scraping it over plain HTTP.** Website is taken, in priority order, from:

1. **YouTube About page** — the link the creator actually published (reused from
   the agent's tested `fetch_youtube_about`, HTTP-only; ScrapingBee only as a
   fallback if YouTube blocks the free GET).
2. **The existing `website_url`** on the row — *only if* it isn't YouTube infra junk.
3. **GPT guess** (`gpt-4o-mini`) from the channel **name** — only when 1 & 2 gave
   nothing (cost deflection). NB: GPT can't use the channel *ID*, only the name,
   and only knows well-known brands — treat as a best-effort fallback.

Then we **scrape that site over HTTP** (homepage + contact/about pages, `mailto:`
+ plaintext emails) — **no Playwright**, so no browser swarm. Also: any email
listed directly on the About page is taken as a strong candidate.

Everything runs in a **thread pool** (sync httpx + litellm) — true parallelism
for I/O, and each result is **source-attributed** so the benchmark shows exactly
where emails come from (About page vs existing site vs GPT-guessed site).

In [ ]:
from __future__ import annotations

import os, re, json, time
from collections import Counter
from datetime import datetime
from urllib.parse import urljoin, urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import sys
sys.path.insert(0, "/Users/utkarshumang/my_projects/ai-agents-service")
sys.path.insert(0, "../..")

from dotenv import load_dotenv
load_dotenv("/Users/utkarshumang/my_projects/lead-enricher-ai-be/.env")

import httpx
import litellm
import tldextract
from bs4 import BeautifulSoup

# Reuse the agent's tested, HTTP-only About-page scraper + link classifier.
from ai_agents.agents.email_finder.nodes.youtube_about_enricher import (
    fetch_youtube_about, _classify_links,
)
import ai_agents.agents.email_finder.nodes.youtube_about_enricher as _yt

from google_utils.google_sheet import GoogleSheetService

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1KywXf2BkClVkW6TMXKp_IQ40l6FGpG_Q9aQKiCRTf2A/edit"
SOURCE_SHEET = "Sheet1"
OUTPUT_SHEET = "Email_Finder_Lite"

CONCURRENCY = 24            # threads; pure I/O so this scales well
HTTP_TIMEOUT = 10.0        # per HTTP GET when scraping a site
MAX_SITE_PAGES = 3         # homepage + up to N contact/about pages
PER_LEAD_BUDGET_S = 35     # soft cap: stop scraping more candidates past this

USE_GPT_GUESS = True       # GPT website guess when About + existing give nothing
GPT_MIN_CONFIDENCE = 0.7   # accept a guess only at/above this self-reported conf

# At 19k scale a single IP gets blocked by YouTube -> the About fetch falls back
# to ScrapingBee (1 credit, ~90s). Set False to stay strictly free (no credits,
# but lower About-page hit-rate). Toggles the reused module's key.
USE_SCRAPINGBEE = True
if not USE_SCRAPINGBEE:
    _yt.SCRAPINGBEE_API_KEY = ""

CHECKPOINT_FILE = "email_finder_youtube_lite_checkpoint.json"
BATCH_SIZE = 100
SKIP_ROWS_WITH_EMAIL = True   # skip rows that already have a real (@) email

## Hygiene helpers (shared with the agent notebook)

In [ ]:
_INFRA_HOSTS = ("ytimg.com","googlevideo.com","youtube.com","youtu.be","google.com",
                "gstatic.com","ggpht.com","googleusercontent.com")
_SOCIAL_HOSTS = ("instagram.com","facebook.com","fb.com","twitter.com","x.com","tiktok.com",
                 "linkedin.com","t.me","snapchat.com","pinterest.com","threads.net",
                 "discord.gg","discord.com","twitch.tv","twitch.com","reddit.com")


def _host(u: str) -> str:
    u = (u or "").strip()
    if not u: return ""
    if "://" not in u: u = "http://" + u
    try: return (urlparse(u).hostname or "").lower()
    except Exception: return ""


def _reg_domain(u: str) -> str:
    ext = tldextract.extract(u or "")
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else (ext.domain or "")


def _is_infra(h: str) -> bool:
    return any(h == d or h.endswith("." + d) for d in _INFRA_HOSTS)


def _is_social(u: str) -> bool:
    h = _host(u)
    return any(h == d or h.endswith("." + d) for d in _SOCIAL_HOSTS)


def sanitize_website(u: str) -> str:
    """Real external website, or '' for YouTube/Google infra junk."""
    h = _host(u)
    if not h or _is_infra(h): return ""
    return (u or "").strip()


def is_real_email(e: str) -> bool:
    e = (e or "").strip().lower()
    return "@" in e and "." in e.split("@")[-1] and e not in {"x","n/a","na",""}


def build_channel_url(handle: str) -> str:
    h = (handle or "").strip()
    if not h: return ""
    if h.startswith("http"): return h
    if h.startswith("UC"): return f"https://www.youtube.com/channel/{h}"
    if h.startswith("@"): return f"https://www.youtube.com/{h}"
    return f"https://www.youtube.com/@{h}"


def _norm_url(u: str) -> str:
    u = (u or "").strip()
    if u and "://" not in u: u = "https://" + u
    return u

## Email extraction + lightweight site scraper (httpx + BeautifulSoup)

In [ ]:
_EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
_MAILTO_RE = re.compile(r"mailto:([^\"\'>?\s]+)", re.I)

# Substrings that mark a junk / placeholder / vendor email (drop these).
_BAD_EMAIL = ("example.com","example.org","sentry","wixpress","godaddy","schema.org",
              "w3.org","yourdomain","yourname","yourcompany","domain.com","email@",
              "user@","test@","name@","placeholder","wix.com","squarespace","cloudflare",
              "no-reply","noreply","donotreply","@sentry","core-js","react","jquery")
_IMG_EXT = (".png",".jpg",".jpeg",".gif",".svg",".webp",".bmp")
_ROLE_PREFIXES = {"info","contact","hello","hi","support","team","admin","sales","press",
                  "media","office","help","hr","jobs","careers","business","booking",
                  "partnerships","inquiries","enquiries","general","mail","email","marketing",
                  "accounts","billing","finance","feedback","social","pr","reception"}
# Legal/system mailboxes that are never a useful outreach contact -> drop entirely.
_DROP_LOCALPARTS = {"privacy","privacypolicy","legal","dmca","abuse","webmaster","postmaster",
                    "hostmaster","copyright","compliance","gdpr","unsubscribe","newsletter",
                    "notifications","noreply","donotreply","mailerdaemon","root","spam","security",
                    "dpo","dataprotection"}
# Third-party/vendor/platform domains embedded in widgets/scripts — never the
# channel's own contact, so drop emails on these outright. (Expanded after v1
# quality analysis surfaced patreon/epidemicsound/etc. repeated across leads.)
# Third-party hosts a creator *links to* (music labels, distributors, game studios,
# merch/payment platforms) — their email is the platform's, never the creator's.
# (Added after the Chloe Ting trace: her About page linked ncs.io song credits and
# we wrongly grabbed NoCopyrightSounds' email.)
_THIRDPARTY_HOSTS = {
    # music labels / distributors
    "ncs.io","distrokid.com","spinnup.com","unitedmasters.com","tunecore.com","cdbaby.com",
    "monstercat.com","soundcloud.com","bandcamp.com","audius.co","awal.com","symphonic.com",
    # game studios / platforms
    "nintendo.com","playstation.com","xbox.com","riotgames.com","ea.com","ubisoft.com",
    "rockstargames.com","epicgames.com","roblox.com",
    # merch / payment / creator / affiliate / gear-list platforms
    "cash.app","venmo.com","ko-fi.com","buymeacoffee.com","cameo.com","represent.com",
    "fanjoy.co","teespring.com","spring.com","bonfire.com","merchbar.com","streamelements.com",
    "streamlabs.com","rightclick.gg","payhip.com","redbubble.com","teepublic.com","spreadshirt.com",
    "threadless.com","kit.co","kit.com","awin.com","deviantart.com","gamersupps.gg","aegm.com",
    "incompetech.com",   # Kevin MacLeod royalty-free music (linked like ncs.io)
    # talent agencies (represent the creator, not the creator's inbox)
    "caa.com","unitedtalent.com","wmeagency.com","gersh.com","abramsartists.com","icmpartners.com",
    # common sponsors whose contact email leaks onto creator sites
    "surfshark.com","nordvpn.com","expressvpn.com","betterhelp.com",
}
# Free-mail providers — personal inboxes; cross-lead frequency is meaningless here,
# so NEVER flag these as third-party (gmail is how most creators are reachable).
_FREEMAIL = {"gmail.com","yahoo.com","outlook.com","hotmail.com","aol.com","icloud.com",
             "proton.me","protonmail.com","gmx.com","gmx.de","mail.com","yandex.com",
             "live.com","msn.com","me.com","ymail.com","googlemail.com"}
_VENDOR_DOMAINS = {
    "amazon.com","amazonaws.com","google.com","gstatic.com","googleapis.com","microsoft.com",
    "apple.com","wordpress.org","wixpress.com","wix.com","sentry.io","cloudflare.com",
    "godaddy.com","squarespace.com","shopify.com","facebook.com","fbcdn.net","w3.org",
    "schema.org","mozilla.org","jquery.com","patreon.com","epidemicsound.com","shopmy.us",
    "geni.us","liketoknow.it","creativecommons.org","medium.com","crowdmade.com","whatnot.com",
    "kick.com","dftba.com","thesoul-publishing.com","stanwith.me","vk-portal.net","bloomberg.net",
    "forkmediagroup.com","insta360.com","stripe.com","paypal.com","mailchimp.com","substack.com",
    "gumroad.com","linktr.ee","beacons.ai","sentry-next.wixpress.com","wp.com","cdn.com",
} | _THIRDPARTY_HOSTS
# Local-parts that are obviously placeholder/demo text.
_PLACEHOLDER = ("your@","you@","youremail","example@","johnappleseed","yourname","yourcompany",
                "sample@","someone@","firstname","lastname","john@example","jane@example",
                "email@example","name@example","user@example")
_BAD_TLDS = {"css","js","json","png","jpg","jpeg","svg","webp","gif","map","scss","ts","woff"}
_CONTACT_HINTS = ("contact","about","team","impressum","kontakt","connect","support","reach")


def _clean_emails(text: str) -> list[str]:
    out, seen = [], set()
    for m in _EMAIL_RE.findall(text or ""):
        e = m.strip().strip(".").lower()
        local, _, dom = e.partition("@")
        if not dom or "." not in dom or len(local) < 2: continue
        if re.search(r"u00[0-9a-f]{2}", local): continue          # JSON-escape artifact (>...)
        if not re.fullmatch(r"[a-z0-9][a-z0-9._%+\-]*", local): continue
        ext = tldextract.extract(dom)
        if not ext.suffix or ext.suffix.split(".")[-1] in _BAD_TLDS: continue
        if any(b in e for b in _BAD_EMAIL) or any(p in e for p in _PLACEHOLDER): continue
        if re.sub(r"[^a-z]", "", local) in _DROP_LOCALPARTS: continue
        if f"{ext.domain}.{ext.suffix}" in _VENDOR_DOMAINS: continue
        if e not in seen:
            seen.add(e); out.append(e)
    return out


def _http_get(client, url):
    try:
        r = client.get(url)
        if r.status_code < 400 and "html" in r.headers.get("content-type", "").lower():
            return r.text
    except Exception:
        return None
    return None


def _emails_and_links(html: str, base: str):
    """Emails from mailto: + VISIBLE TEXT only (script/style/head stripped, so the
    JSON `\u003e...@vendor.com` boilerplate that polluted v1 is gone), plus the
    same-site contact/about links worth following."""
    soup = BeautifulSoup(html or "", "html.parser")
    base_host = urlparse(base).netloc
    mailtos, sublinks = [], []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().startswith("mailto:"):
            mailtos.append(href.split(":", 1)[1].replace("%40", "@").split("?")[0])
        else:
            blob = (href + " " + (a.get_text() or "")).lower()
            if any(h in blob for h in _CONTACT_HINTS):
                full = urljoin(base, href)
                if urlparse(full).netloc == base_host and full not in sublinks:
                    sublinks.append(full)
    for tag in soup(["script", "style", "noscript", "template", "svg", "head"]):
        tag.decompose()
    text = soup.get_text(" ")
    mailto_clean = set(_clean_emails(" ".join(mailtos)))
    emails = _clean_emails(" ".join(mailtos) + " " + text)
    emails.sort(key=lambda e: e not in mailto_clean)   # mailto hits are most reliable
    return emails, sublinks


def scrape_site_emails(website: str, max_pages: int = MAX_SITE_PAGES,
                       timeout: float = HTTP_TIMEOUT) -> list[str]:
    """Homepage + a few contact/about pages -> cleaned emails (no browser)."""
    website = _norm_url(website)
    if not website: return []
    out, seen = [], set()
    def _add(es):
        for e in es:
            if e not in seen:
                seen.add(e); out.append(e)
    headers = {"User-Agent": "Mozilla/5.0 (compatible; EmailFinderLite/1.0)"}
    try:
        with httpx.Client(timeout=timeout, follow_redirects=True, headers=headers) as client:
            html = _http_get(client, website)
            if not html: return out
            emails, sublinks = _emails_and_links(html, website)
            _add(emails)
            for link in sublinks[:max_pages]:
                h2 = _http_get(client, link)
                if h2:
                    e2, _ = _emails_and_links(h2, link); _add(e2)
    except Exception:
        pass
    return out


def _name_tokens(name: str) -> list[str]:
    return [t for t in re.split(r"[^a-z0-9]+", (name or "").lower()) if len(t) >= 3]


def pick_best_email(emails: list[str], website: str, name: str) -> dict | None:
    if not emails: return None
    site_dom = _reg_domain(website) if website else ""
    toks = _name_tokens(name)
    scored = []
    for e in emails:
        local, _, dom = e.partition("@")
        edom = _reg_domain(dom)
        score = 0.4
        domain_match = bool(site_dom) and edom == site_dom
        if domain_match: score += 0.35
        name_match = any(t in local for t in toks)
        if name_match: score += 0.20
        is_role = local in _ROLE_PREFIXES
        if is_role: score -= 0.10
        scored.append((score, e, domain_match, is_role, name_match))
    scored.sort(reverse=True)
    s = scored[0]
    return {"email": s[1], "confidence": round(min(s[0], 0.95), 2),
            "domain_match": s[2], "is_role": s[3], "name_match": s[4]}

## GPT website guess (litellm gpt-4o-mini — same model as the agent)

In [ ]:
_GUESS_PROMPT = """You are given a YouTube channel's name and details. If you are CONFIDENT, \
from your own training knowledge, that you know their OWN official personal or company website, \
return it. If you are not confident, return null — do not guess or invent a domain. Do not \
return a social-media profile or a directory page as their website.

Name: {name}
{context}

Return ONLY JSON: {{"website": "https://... or null", "confidence": 0.0 to 1.0}}"""


def gpt_guess_website(name: str, row: dict) -> tuple[str | None, float]:
    if not name: return None, 0.0
    ctx = []
    for k in ("niche", "category", "country"):
        v = str(row.get(k) or "").strip()
        if v: ctx.append(f"{k}: {v}")
    context = "\n".join(ctx) or "(no extra details)"
    try:
        resp = litellm.completion(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": _GUESS_PROMPT.format(name=name, context=context)}],
            response_format={"type": "json_object"},
            temperature=0,
        )
        data = json.loads(resp.choices[0].message.content)
        w = data.get("website")
        if not isinstance(w, str) or not w.lower().startswith("http"):
            return None, 0.0
        try: conf = float(data.get("confidence", 0) or 0)
        except (TypeError, ValueError): conf = 0.0
        if conf < GPT_MIN_CONFIDENCE or _is_social(w):
            return None, conf
        return w.strip(), conf
    except Exception:
        return None, 0.0

## v2: link-in-bio / shortener resolution

The biggest miss bucket was "site found, no email" — because the About-page link
was often a **link-in-bio aggregator** (Linktree/Beacons) or a **shortener**
(bit.ly/amzn.to/sjv.io), not the real homepage. `expand_candidates()` follows
those hops to the creator's actual site(s) and harvests any email on the
aggregator page itself. Measured lift: ~14% recovery on previously-empty leads.

In [ ]:
# Link-in-bio aggregators (fetch -> harvest their outbound links + page emails)
_AGG_HOSTS = {"linktr.ee","beacons.ai","lnk.bio","linkin.bio","bio.link","allmylinks.com",
              "campsite.bio","solo.to","tap.bio","koji.to","withkoji.com","msha.ke","flowpage.com",
              "snipfeed.co","link.space","hoo.be","komi.io","shor.by","liinks.co","carrd.co",
              "pillar.io","glnk.io","znap.link"}
# URL shorteners / affiliate redirectors (resolve to final URL)
_SHORT_HOSTS = {"bit.ly","tinyurl.com","t.co","goo.gl","ow.ly","buff.ly","rb.gy","cutt.ly",
                "shorturl.at","rebrand.ly","is.gd","tiny.cc","sjv.io","pxf.io","go.magik.ly","lnk.to"}
# Marketplaces/landing hosts that never carry the creator's contact -> skip as targets
_MARKET_HOSTS = {"amazon.com","amzn.to","apps.apple.com","play.google.com","etsy.com","patreon.com",
                 "geni.us","fanlink.to","ffm.to","smarturl.it","spotify.com","itunes.apple.com"}


def _unescape(s):
    return (s or "").replace("\\u002f", "/").replace("\\u002F", "/").replace("\\/", "/")


def _urls_in_html(html):
    """All http(s) URLs in the HTML (incl. JSON blobs) — used to mine aggregator pages."""
    h = _unescape(html)
    for ch in ('"', "'", "<", ">", "\\", "(", ")"):
        h = h.replace(ch, " ")
    return [t.rstrip(".,;") for t in h.split() if t.startswith("http")]


def _root_url(u):
    """Normalize to the site root so we scrape the homepage/contact, not a deep
    page (Chloe Ting: we had chloeting.com but scraped a deep /program/ URL)."""
    try:
        p = urlparse(u if "://" in u else "https://" + u)
        return f"{p.scheme}://{p.netloc}/" if p.netloc else u
    except Exception:
        return u


def _name_related(u, toks):
    rd = _reg_domain(u)
    return any(t in rd for t in toks)


def _bucket(u):
    h = _host(u); rd = _reg_domain(u)
    if not h: return "bad"
    if _is_social(u): return "social"
    if rd in _AGG_HOSTS or h in _AGG_HOSTS: return "agg"
    if rd in _SHORT_HOSTS or h in _SHORT_HOSTS: return "short"
    # third-party (music labels/games/merch/payment) — linked but not the creator's
    if rd in _THIRDPARTY_HOSTS or h in _THIRDPARTY_HOSTS: return "thirdparty"
    if rd in _MARKET_HOSTS or h in _MARKET_HOSTS: return "market"
    if _is_infra(h): return "infra"
    return "real"


def expand_candidates(links, client, name=""):
    """Turn raw About-page links into real website targets + any aggregator-page
    emails. Resolves shortener redirects, mines link-in-bio pages, drops third-party
    domains, normalizes to root, and prioritizes domains that match the channel name."""
    toks = _name_tokens(name)
    real, aggs, shorts, agg_emails = [], [], [], []
    for u in links:
        b = _bucket(u)
        if b == "real": real.append(u)
        elif b == "agg": aggs.append(u)
        elif b == "short": shorts.append(u)
    for s in shorts[:6]:
        try:
            fu = str(client.get(s).url); b = _bucket(fu)
            if b == "real": real.append(fu)
            elif b == "agg": aggs.append(fu)
        except Exception:
            pass
    for a in aggs[:3]:
        try:
            html = client.get(a).text
            agg_emails += _clean_emails(html)
            real += [u for u in _urls_in_html(html) if _bucket(u) == "real"]
        except Exception:
            pass
    seen, out = set(), []
    for u in real:
        d = _reg_domain(u)
        if d and d not in seen:
            seen.add(d); out.append(_root_url(u))
    # scrape name-matching domains first (chloeting.com before a credited 3rd-party)
    out.sort(key=lambda u: 0 if _name_related(u, toks) else 1)
    return out[:4], agg_emails

## The per-channel pipeline (v2)

Gathers ALL About-page links + existing site, **resolves hops** to real
destinations, scrapes each (capped by `PER_LEAD_BUDGET_S`), and falls back to
About-page / aggregator emails, then GPT-guess. Records where it came from.

In [ ]:
def find_email_lite(row: dict) -> dict:
    t0 = time.monotonic()
    name = (row.get("title") or "").strip()
    channel_url = build_channel_url((row.get("channel_handle") or "").strip())
    existing_site = sanitize_website(row.get("website_url"))
    existing_email = row.get("email") if is_real_email(row.get("email")) else ""

    log = []
    about_emails, about_links = [], []
    if channel_url:
        try:
            about = fetch_youtube_about(channel_url)
            about_links = about.get("links", [])
            about_emails = _clean_emails(" ".join(about.get("emails", [])))
            log.append(f"about(links={len(about_links)},email={len(about_emails)})")
        except Exception as e:
            log.append(f"about_err:{str(e)[:30]}")

    seed = list(about_links) + ([existing_site] if existing_site else [])
    pool = list(about_emails)
    cands = []
    headers = {"User-Agent": "Mozilla/5.0 (compatible; EmailFinderLite/1.0)"}
    try:
        with httpx.Client(timeout=HTTP_TIMEOUT, follow_redirects=True, headers=headers) as client:
            cands, agg_emails = expand_candidates(seed, client, name)
            pool += agg_emails
    except Exception as e:
        log.append(f"expand_err:{str(e)[:30]}")
    log.append(f"cands={len(cands)}")

    # GPT website guess only when nothing else surfaced
    if USE_GPT_GUESS and not cands and not pool:
        gsite, gconf = gpt_guess_website(name, row)
        if gsite:
            cands = [gsite]; log.append(f"gpt={gsite}({gconf})")

    best = website_used = website_source = None
    # 1) scrape resolved candidate sites (best signal)
    for w in cands:
        try: emails = scrape_site_emails(w)
        except Exception: emails = []
        if emails:
            best = pick_best_email(emails, w, name)
            website_used = w
            website_source = "about_page" if _bucket(w) == "real" else _bucket(w)
            break
        if time.monotonic() - t0 > PER_LEAD_BUDGET_S:
            break
    # 2) else use any About-page / aggregator email collected
    if not best and pool:
        best = pick_best_email(pool, "", name)
        website_source = "about_or_aggregator_email"
    # 3) last resort: a real email already on the row
    if not best and existing_email:
        best = {"email": existing_email, "confidence": 0.5, "domain_match": False,
                "is_role": False, "name_match": False}
        website_used, website_source = existing_site, "row"

    source = (f"site:{website_source}" if website_used else (website_source or "none")) if best else "none"
    # name_related: does the email's domain or local-part echo the channel name?
    # (chloeting.com / chloe@... -> True; hello@ncs.io for "Chloe Ting" -> False).
    em = best["email"] if best else ""
    toks = _name_tokens(name)
    nrel = bool(em) and (any(t in em.split("@")[0] for t in toks)
                         or any(t in _reg_domain(em.split("@")[-1]) for t in toks))
    return {
        "name": name,
        "channel_url": channel_url,
        "email": em,
        "confidence": best["confidence"] if best else 0.0,
        "source": source,
        "website_used": website_used or "",
        "website_source": website_source or "",
        "domain_match": best.get("domain_match") if best else False,
        "name_related": nrel,
        "is_role": best.get("is_role") if best else False,
        "status": "email_found" if best else "not_found",
        "n_candidates": len(cands),
        "about_emails_n": len(about_emails),
        "log": "; ".join(log),
        "_elapsed_s": round(time.monotonic() - t0, 1),
        "_dedup_key": row.get("_dedup_key", ""),
    }

In [ ]:
# ── Threaded runner (pure I/O -> threads parallelize cleanly, no browsers) ───
LEAD_TIMINGS = []

def run_batch_lite(rows, concurrency=CONCURRENCY, verbose=True):
    total = len(rows)
    results = [None] * total
    done = 0
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futs = {ex.submit(find_email_lite, r): i for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            i = futs[fut]
            try:
                res = fut.result()
            except Exception as e:
                res = {"name": rows[i].get("title", ""), "email": "", "confidence": 0.0,
                       "source": "error", "status": "failed", "log": str(e)[:120],
                       "_elapsed_s": 0.0, "_dedup_key": rows[i].get("_dedup_key", ""),
                       "website_used": "", "website_source": "", "is_role": False,
                       "domain_match": False, "n_candidates": 0, "about_emails_n": 0}
            results[i] = res
            LEAD_TIMINGS.append(res.get("_elapsed_s", 0))
            done += 1
            if verbose:
                print(f"  [{done}/{total}] {res['status']:11} {res['_elapsed_s']:>5}s  "
                      f"{(res['email'] or '(none)'):30} <{res['source']:18}> {res['name'][:24]}")
    return results

## Load rows + checkpoint

In [ ]:
def collect_rows(sheet_service, spreadsheet_id, already_processed):
    ok, df = sheet_service.get_sheet_data(spreadsheet_id, SOURCE_SHEET)
    assert ok, f"Failed to read sheet: {df}"
    stats = Counter()
    seen = set(already_processed)
    rows = []
    for _, r in df.iterrows():
        stats["total"] += 1
        rd = r.to_dict()
        if SKIP_ROWS_WITH_EMAIL and is_real_email(rd.get("email")):
            stats["has_email_skipped"] += 1; continue
        key = (rd.get("channel_handle") or "").strip() or (rd.get("title") or "").strip()
        if not key:
            stats["no_key"] += 1; continue
        if key in seen:
            stats["dup_or_processed"] += 1; continue
        seen.add(key); rd["_dedup_key"] = key; rows.append(rd)
    print(f"Collection: total={stats['total']} | had_email={stats['has_email_skipped']} | "
          f"no_key={stats['no_key']} | dup/done={stats['dup_or_processed']} | to_process={len(rows)}")
    return rows


def load_checkpoint(sid):
    if os.path.exists(CHECKPOINT_FILE):
        d = json.load(open(CHECKPOINT_FILE))
        if d.get("spreadsheet_id") == sid:
            print(f"Resuming — {len(d['processed_keys'])} already done")
            return d
    return {"spreadsheet_id": sid, "processed_keys": [], "found": 0, "not_found": 0, "last_updated": None}


def save_checkpoint(cp):
    cp["last_updated"] = datetime.now().isoformat()
    json.dump(cp, open(CHECKPOINT_FILE, "w"), indent=2)


def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE); print("Checkpoint cleared.")

In [ ]:
# ── Output sheet ─────────────────────────────────────────────────────────────
# name_related is the LAST column on purpose: appending (not inserting mid-schema)
# keeps every pre-existing 13-column row aligned when the header grows to 14.
OUTPUT_HEADERS = ["name","channel_url","email","confidence","source","website_used",
                  "website_source","domain_match","is_role","status","about_emails_n",
                  "log","timestamp","name_related"]


def result_to_row(res: dict) -> list:
    return [res.get("name",""), res.get("channel_url",""), res.get("email",""),
            str(res.get("confidence","")), res.get("source",""), res.get("website_used",""),
            res.get("website_source",""), str(res.get("domain_match","")),
            str(res.get("is_role","")), res.get("status",""),
            str(res.get("about_emails_n","")), res.get("log",""),
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"), str(res.get("name_related",""))]


def _ensure_output(sheet_service, sid, name):
    ok, names = sheet_service.list_sheets(sid)
    if not (ok and name in names):
        sheet_service.service.spreadsheets().batchUpdate(
            spreadsheetId=sid, body={"requests": [{"addSheet": {"properties": {"title": name}}}]}).execute()
        print(f"Created tab '{name}'")
    ok, vals = sheet_service.get_sheet_values(sid, f"{name}!A1:N1")
    if not ok or not vals:
        sheet_service.append_rows(sid, name, [OUTPUT_HEADERS]); print(f"Added headers to {name}")
    elif len(vals[0]) < len(OUTPUT_HEADERS):
        # upgrade an older/narrower header in place (e.g. pre-name_related tabs)
        end = chr(64 + len(OUTPUT_HEADERS))
        sheet_service.service.spreadsheets().values().update(
            spreadsheetId=sid, range=f"{name}!A1:{end}1",
            valueInputOption="RAW", body={"values": [OUTPUT_HEADERS]}).execute()
        print(f"Upgraded {name} header to {len(OUTPUT_HEADERS)} cols")

In [ ]:
def run_pipeline(spreadsheet_id, limit=None, batch_size=BATCH_SIZE, dry_run=True):
    """Find emails over SOURCE_SHEET; write every processed channel to OUTPUT_SHEET.
    Resumable via CHECKPOINT_FILE (keyed by channel handle)."""
    ss = GoogleSheetService()
    cp = load_checkpoint(spreadsheet_id)
    rows = collect_rows(ss, spreadsheet_id, set(cp["processed_keys"]))
    if limit is not None:
        rows = rows[:limit]; print(f"  (limited to {len(rows)})")
    if not rows:
        print("Nothing to process."); return cp
    if not dry_run:
        _ensure_output(ss, spreadsheet_id, OUTPUT_SHEET)

    src_stats = Counter()
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        print(f"\nBatch {start//batch_size + 1} — {len(batch)} channels")
        results = run_batch_lite(batch)
        out_rows, out_keys = [], []
        for res in results:
            if res["status"] == "email_found":
                cp["found"] += 1; src_stats[res["source"]] += 1
            else:
                cp["not_found"] += 1
            out_rows.append(result_to_row(res)); out_keys.append(res["_dedup_key"])
        if dry_run:
            print("  [dry_run] not writing. sample:")
            for r in out_rows[:5]:
                print("   ", r[0][:24], "|", r[2] or "(none)", "|", r[4])
            cp["processed_keys"].extend(out_keys); continue
        ok, msg = ss.append_rows(spreadsheet_id, OUTPUT_SHEET, out_rows)
        print(f"  -> {msg}")
        if ok:
            cp["processed_keys"].extend(out_keys); save_checkpoint(cp)

    print(f"\n{'='*46}\nFound: {cp['found']} | Not found: {cp['not_found']}")
    if src_stats:
        print("Email sources:")
        for s, c in src_stats.most_common(): print(f"  {s}: {c}")
    return cp

## Preview — access + counts

In [ ]:
ss = GoogleSheetService()
spreadsheet_id = ss.extract_spreadsheet_id(SPREADSHEET_URL)
ok, df = ss.get_sheet_data(spreadsheet_id, SOURCE_SHEET)
assert ok, df
need = sum(0 if is_real_email(e) else 1 for e in df["email"].tolist())
print(f"'{SOURCE_SHEET}': {len(df)} rows | already have email: {len(df)-need} | need email: {need}")

## Single-channel smoke test

In [ ]:
sample = {"title": "Marques Brownlee", "channel_handle": "@mkbhd", "email": "x",
          "website_url": "https://i.ytimg.com/vi/x/hqdefault.jpg", "niche": "tech", "country": "US"}
import pprint; pprint.pprint(find_email_lite(sample))

## Benchmark — 50 rows with SOURCE BREAKDOWN

Run this first. It shows real per-lead timing, the found-rate, an ETA for 17.3k,
and — the key question — **where the found emails come from** (About-page email,
About-page site, existing site, or GPT-guessed site). That tells us whether the
GPT-guess path earns its keep or whether About-page is the real backbone.

In [ ]:
import statistics

BENCH_N = 50
# The sheet is sorted by subscriber count, so the TOP rows are mega-channels
# (label-managed, no scrapeable email) — a misleading sample. Stride evenly across
# the whole needs-email list to include the long tail, which is the real target.
_all_need = collect_rows(ss, spreadsheet_id, set())
_step = max(1, len(_all_need) // BENCH_N)
bench_rows = _all_need[::_step][:BENCH_N]
print(f"\nBenchmarking {len(bench_rows)} rows sampled across {len(_all_need)} "
      f"(every {_step}th, concurrency={CONCURRENCY})...\n")

LEAD_TIMINGS.clear()
t0 = time.monotonic()
res = run_batch_lite(bench_rows)
wall = time.monotonic() - t0

found = [r for r in res if r["status"] == "email_found"]
by_source = Counter(r["source"] for r in found)
by_websrc = Counter(r["website_source"] for r in found)
ts = sorted(LEAD_TIMINGS) or [0]
def _p(p): return ts[min(len(ts)-1, int(len(ts)*p))]

print(f"\n{'='*52}\nLITE benchmark: {len(res)} rows @ concurrency={CONCURRENCY}\n{'='*52}")
print(f"  Wall clock:       {wall:.0f}s  ({wall/len(res):.1f}s/row effective)")
print(f"  Per-lead seconds: min={ts[0]:.0f} median={statistics.median(ts):.0f} p90={_p(0.9):.0f} max={ts[-1]:.0f}")
print(f"  Emails found:     {len(found)}/{len(res)}  ({100*len(found)/len(res):.0f}%)")
print(f"  Role/generic:     {sum(1 for r in found if r['is_role'])}  | domain-matched: {sum(1 for r in found if r['domain_match'])}")
print(f"  Found by source:  {dict(by_source.most_common())}")
print(f"  Found by website: {dict(by_websrc.most_common())}")
print(f"\n  ETA for ~17,300 rows @ concurrency={CONCURRENCY}: ~{(wall/len(res))*17300/3600:.1f} h")
print("  (raise CONCURRENCY to cut ~linearly; it's all I/O)")

## Quality re-filter of an EXISTING output tab (no re-scrape)

Applies the hardened junk filters + a cross-lead boilerplate drop to a tab you
already produced, and writes a `*_Clean` tab. This only *removes* junk from what's
there. To also *recover* real emails that v1 missed under the junk, do a full
re-run with the fixed extractor (`clear_checkpoint()` then `run_pipeline(...)`).

In [ ]:
def refilter_output(spreadsheet_id, src_tab=OUTPUT_SHEET, max_crosslead=2):
    ss = GoogleSheetService()
    ok, df = ss.get_sheet_data(spreadsheet_id, src_tab); assert ok, df
    has = df[df["email"].astype(str).str.contains("@", na=False)]
    freq = Counter(has["email"].str.lower())
    kept, dropped = 0, Counter()
    rows_out = [list(df.columns)]
    for _, r in df.iterrows():
        e = str(r["email"]).strip().lower()
        if not e or "@" not in e:
            continue
        if not _clean_emails(e):
            dropped["junk_filter"] += 1; continue
        if freq[e] > max_crosslead:
            dropped["cross_lead_boilerplate"] += 1; continue
        rows_out.append([r[c] for c in df.columns]); kept += 1

    clean_tab = src_tab + "_Clean"
    ok, names = ss.list_sheets(spreadsheet_id)
    if clean_tab not in names:
        ss.service.spreadsheets().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body={"requests": [{"addSheet": {"properties": {"title": clean_tab}}}]}).execute()
    print(f"kept {kept} clean of {len(has)} found | dropped {dict(dropped)}")
    print(ss.clear_and_rewrite_sheet(spreadsheet_id, clean_tab, rows_out)[1])
    return kept


# refilter_output(spreadsheet_id)   # -> writes Email_Finder_Lite_Clean

## Precision audit — flag/remove unrelated-domain emails

Finds emails that almost certainly belong to a third party, not the creator: a
domain on the third-party/vendor blocklist, OR a domain that yields emails for
many *unrelated* channels (cross-lead frequency) while not matching the channel
name. `remove=True` rewrites the tabs without them; `clean_sheet1=True` blanks
them from Sheet1 too. (This is what catches the Chloe Ting -> ncs.io case.)

In [ ]:
def audit_thirdparty(spreadsheet_id, tabs=("Email_Finder_Lite_Clean","Email_Finder_Lite_Recovered"),
                     min_channels=3, remove=False, clean_sheet1=False, remove_review=False):
    import pandas as pd
    ss = GoogleSheetService()
    frames = []
    for t in tabs:
        ok, df = ss.get_sheet_data(spreadsheet_id, t)
        if ok and len(df):
            df = df[df["email"].astype(str).str.contains("@", na=False)].copy()
            df["_tab"] = t; frames.append(df)
    if not frames:
        print("no data"); return
    allr = pd.concat(frames, ignore_index=True)
    allr["_dom"] = allr["email"].map(lambda e: _reg_domain(str(e).split("@")[-1]))
    dom_freq = allr.groupby("_dom")["name"].nunique()      # distinct channels per domain
    cross = set(dom_freq[dom_freq >= min_channels].index)

    def flag(row):
        dom = row["_dom"]; toks = _name_tokens(str(row["name"]))
        if dom in _FREEMAIL: return ""                      # personal inbox -> always keep
        if any(t in dom for t in toks): return ""           # name-related -> keep
        if dom in _VENDOR_DOMAINS: return "thirdparty"      # blocklist -> safe to remove
        if dom in cross: return "review"                    # cross-lead, non-free -> review
        return ""
    allr["_flag"] = allr.apply(flag, axis=1)
    tp = allr[allr["_flag"] == "thirdparty"]
    rv = allr[allr["_flag"] == "review"]
    print(f"audited {len(allr)} emails | thirdparty(remove) {len(tp)} | cross-lead(review) {len(rv)}")
    print(" -- thirdparty (auto-removed when remove=True):")
    for d, c in Counter(tp["_dom"]).most_common(12): print(f"     {c:3} ch  {d}")
    print(" -- cross-lead review (kept unless remove_review=True):")
    for d, c in Counter(rv["_dom"]).most_common(12): print(f"     {c:3} ch  {d}")
    flagged = tp if not remove_review else pd.concat([tp, rv])

    if remove:
        bad = set(zip(flagged["_tab"], flagged["email"].str.lower()))
        for t in tabs:
            ok, df = ss.get_sheet_data(spreadsheet_id, t)
            if not (ok and len(df)): continue
            keep = [list(df.columns)]; removed = 0
            for _, r in df.iterrows():
                if (t, str(r["email"]).strip().lower()) in bad: removed += 1; continue
                keep.append([r[c] for c in df.columns])
            ss.clear_and_rewrite_sheet(spreadsheet_id, t, keep)
            print(f"   {t}: removed {removed}, kept {len(keep)-1}")

    if clean_sheet1:
        badmails = set(flagged["email"].str.lower())
        ok, vals = ss.get_sheet_values(spreadsheet_id, "Sheet1"); header = vals[0]
        ei = header.index("email")
        si = header.index("email_source") if "email_source" in header else None
        ci = header.index("email_confidence") if "email_confidence" in header else None
        def cell(row, i): return row[i] if (i is not None and len(row) > i) else ""
        cE=[[header[ei]]]; cS=[["email_source"]]; cC=[["email_confidence"]]; blanked=0
        for row in vals[1:]:
            if cell(row, ei).strip().lower() in badmails:
                cE.append([""]); cS.append([""]); cC.append([""]); blanked += 1
            else:
                cE.append([cell(row, ei)]); cS.append([cell(row, si)]); cC.append([cell(row, ci)])
        def L(i):
            s=""; i+=1
            while i: i,r=divmod(i-1,26); s=chr(65+r)+s
            return s
        total=len(vals); svc=ss.service.spreadsheets().values()
        svc.update(spreadsheetId=spreadsheet_id, range=f"Sheet1!{L(ei)}1:{L(ei)}{total}",
                   valueInputOption="RAW", body={"values": cE}).execute()
        if si is not None:
            svc.update(spreadsheetId=spreadsheet_id, range=f"Sheet1!{L(si)}1:{L(si)}{total}",
                       valueInputOption="RAW", body={"values": cS}).execute()
        if ci is not None:
            svc.update(spreadsheetId=spreadsheet_id, range=f"Sheet1!{L(ci)}1:{L(ci)}{total}",
                       valueInputOption="RAW", body={"values": cC}).execute()
        print(f"   Sheet1: blanked {blanked} suspect emails")
    return flagged


# audit_thirdparty(spreadsheet_id)                                  # report only
# audit_thirdparty(spreadsheet_id, remove=True, clean_sheet1=True)  # apply

## Targeted recovery run — re-process only the "site-found, no email" leads

Uses the v2 resolver on exactly the leads where v1 found a website but no email
(`status=not_found` AND `log` shows links). Recovered emails are written to
`Email_Finder_Lite_Recovered`; pass `backfill=True` to also write them into
Sheet1 (junk/empty rows only) with provenance.

In [ ]:
def recovery_rows(spreadsheet_id, prior_tab="Email_Finder_Lite"):
    """Sheet1 rows for leads that previously found a site but no email."""
    ss = GoogleSheetService()
    ok, out = ss.get_sheet_data(spreadsheet_id, prior_tab); assert ok, out
    ok, df = ss.get_sheet_data(spreadsheet_id, "Sheet1"); assert ok, df
    by_handle = {}
    for _, r in df.iterrows():
        by_handle[(str(r.get("title", "")).strip())] = r.to_dict()
    targets = out[(out["status"] == "not_found")
                  & (out["log"].astype(str).str.contains("links=[1-9]", regex=True))]
    rows = []
    for _, r in targets.iterrows():
        src = by_handle.get(str(r["name"]).strip())
        if src and not is_real_email(src.get("email")):
            src = dict(src); src["_dedup_key"] = str(r["name"]).strip()
            rows.append(src)
    print(f"recovery targets (site-found, still blank): {len(rows)}")
    return rows


def backfill_sheet1(spreadsheet_id, src_tab):
    """Write a found-emails tab back into Sheet1's email column (junk rows only),
    reusing the email_source/email_confidence provenance columns."""
    ss = GoogleSheetService()
    ok, rec = ss.get_sheet_data(spreadsheet_id, src_tab); assert ok, rec
    lut = {}
    for _, r in rec.iterrows():
        cu = str(r["channel_url"]).strip().lower(); em = str(r["email"]).strip()
        if cu and is_real_email(em):
            lut.setdefault(cu, (em, str(r["source"]), str(r["confidence"])))
    ok, vals = ss.get_sheet_values(spreadsheet_id, "Sheet1"); assert ok, vals
    header = vals[0]; ncols = len(header)
    ei = header.index("email")
    si = header.index("email_source") if "email_source" in header else ncols
    ci = header.index("email_confidence") if "email_confidence" in header else (si + 1)
    data = vals[1:]; N = len(data)
    def cell(row, i): return row[i] if len(row) > i else ""
    col_e = [[header[ei]]]; col_s = [["email_source"]]; col_c = [["email_confidence"]]
    filled = 0
    for row in data:
        handle = row[1] if len(row) > 1 else ""
        cur = cell(row, ei); cu = build_channel_url(handle).lower()
        if (not is_real_email(cur)) and cu in lut:
            em, s, cf = lut[cu]; col_e.append([em]); col_s.append([s]); col_c.append([cf]); filled += 1
        else:
            col_e.append([cur]); col_s.append([cell(row, si)]); col_c.append([cell(row, ci)])
    def L(i):
        s = ""; i += 1
        while i: i, r = divmod(i - 1, 26); s = chr(65 + r) + s
        return s
    total = N + 1; svc = ss.service.spreadsheets().values()
    for rng, body in ((f"Sheet1!{L(ei)}1:{L(ei)}{total}", col_e),
                      (f"Sheet1!{L(si)}1:{L(si)}{total}", col_s),
                      (f"Sheet1!{L(ci)}1:{L(ci)}{total}", col_c)):
        svc.update(spreadsheetId=spreadsheet_id, range=rng, valueInputOption="RAW",
                   body={"values": body}).execute()
    print(f"backfilled {filled} new emails into Sheet1")
    return filled


def run_recovery(spreadsheet_id, limit=None, batch_size=200, backfill=False):
    ss = GoogleSheetService()
    rows = recovery_rows(spreadsheet_id)
    if limit is not None:
        rows = rows[:limit]; print(f"  (limited to {len(rows)})")
    rec_tab = "Email_Finder_Lite_Recovered"
    _ensure_output(ss, spreadsheet_id, rec_tab)
    recovered = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        print(f"\nBatch {start//batch_size + 1} — {len(batch)} leads")
        results = run_batch_lite(batch)
        hits = [r for r in results if r["status"] == "email_found"]
        recovered += len(hits)
        if hits:
            ss.append_rows(spreadsheet_id, rec_tab, [result_to_row(r) for r in hits])
            print(f"  -> wrote {len(hits)} recovered (running total {recovered})")
    print(f"\nRECOVERED {recovered} new emails of {len(rows)} targets "
          f"({100*recovered/max(len(rows),1):.0f}%)")
    if backfill:
        print("backfilling Sheet1 from", rec_tab, "...")
        backfill_sheet1(spreadsheet_id, rec_tab)
    return recovered

## Test run (writes to Email_Finder_Lite)

In [ ]:
run_pipeline(spreadsheet_id, limit=20, dry_run=True)
# run_pipeline(spreadsheet_id, limit=20, dry_run=False)   # then write

In [ ]:
# Full / scaled run (resumable):
# run_pipeline(spreadsheet_id, limit=None, dry_run=False)
# clear_checkpoint()

## ▶ Targeted recovery — RUN THIS (the enhanced v2 re-pass)

Re-processes ONLY the ~9,443 "site-found, no-email" leads with the v2 resolver
(link-in-bio/shortener resolution, third-party blocklist, name-prioritized
scraping). Writes hits to `Email_Finder_Lite_Recovered`; `backfill=True` also
writes them into Sheet1's email column (junk rows only) with provenance.

Requires the setup cells above to have run (esp. the **Preview** cell, which
defines `spreadsheet_id`). Start with the small validation call, then go full.

In [ ]:
# 1) Validation — ~100 leads, no writes (sanity check the recovery rate)
run_recovery(spreadsheet_id, limit=100, backfill=False)

In [ ]:
# 2) FULL recovery on all site-found leads + backfill Sheet1 (~1.5-2 hr, resumable-ish)
run_recovery(spreadsheet_id, backfill=True)

In [ ]:
# 3) After the run — sweep any new third-party hits, then apply
# audit_thirdparty(spreadsheet_id)                                  # report only
# audit_thirdparty(spreadsheet_id, remove=True, clean_sheet1=True)  # apply